# FlashAttention Walkthrough

Goal: understand FlashAttention as an IO-aware exact attention algorithm before trying to build CUDA kernels.

You should be able to explain why materializing the full `N x N` score matrix is expensive, how online softmax keeps exact results block by block, and where the official implementation stores Python interfaces and CUDA kernels.


## Paper-to-code map

Official source:

- `learning/kernel-engineering/official/repos/flash-attention/flash_attn/flash_attn_interface.py`
- `learning/kernel-engineering/official/repos/flash-attention/csrc/flash_attn/flash_api.cpp`
- `learning/kernel-engineering/official/repos/flash-attention/hopper/softmax.h`

Runnable local source:

- `learning/kernel-engineering/src/flash_attention.py`
- `learning/kernel-engineering/src/tests/test_all.py`


In [ ]:
from __future__ import annotations
import importlib.util
import math
import platform
from pathlib import Path
import random
import torch
REPO = Path.cwd()
if not (REPO / "learning").exists():
    REPO = Path.cwd().parent
print("python", platform.python_version())
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
print("repo", REPO)


## Step 1: naive attention memory shape

Naive attention explicitly materializes scores and probabilities:

```text
Q, K, V: (N, d)
S = Q K^T: (N, N)
P = softmax(S): (N, N)
O = P V: (N, d)
```

The `N x N` objects dominate memory as sequence length grows.


In [ ]:
def naive_attention(Q, K, V):
    N, d = len(Q), len(Q[0])
    scale = 1.0 / math.sqrt(d)
    S = [[sum(Q[i][k] * K[j][k] for k in range(d)) * scale for j in range(N)] for i in range(N)]
    P = []
    for row in S:
        m = max(row)
        e = [math.exp(x - m) for x in row]
        z = sum(e)
        P.append([x / z for x in e])
    return [[sum(P[i][j] * V[j][k] for j in range(N)) for k in range(d)] for i in range(N)]

for N in [128, 1024, 8192]:
    score_bytes = N * N * 2
    print({"N": N, "fp16_score_MB": round(score_bytes / 1024**2, 2)})


## Step 2: online softmax update for one row

For each query row, FlashAttention streams K/V blocks and keeps:

```text
m: running max
l: running normalizer
 o: running unnormalized output numerator
```

When a new block has a larger max, previous numerator and denominator are rescaled.


In [ ]:
def flash_attention_row(q, K, V, block_n=3):
    d = len(q)
    scale = 1.0 / math.sqrt(d)
    m = -math.inf
    l = 0.0
    o = [0.0] * d
    trace = []
    for start in range(0, len(K), block_n):
        K_block = K[start:start + block_n]
        scores = [sum(q[t] * key[t] for t in range(d)) * scale for key in K_block]
        m_new = max(m, max(scores))
        rescale = math.exp(m - m_new) if m != -math.inf else 0.0
        l *= rescale
        o = [x * rescale for x in o]
        for offset, score in enumerate(scores):
            p = math.exp(score - m_new)
            l += p
            v = V[start + offset]
            for t in range(d):
                o[t] += p * v[t]
        m = m_new
        trace.append({"block_start": start, "m": round(m, 4), "l": round(l, 4)})
    return [x / l for x in o], trace

random.seed(7)
N, d = 8, 4
Q = [[random.random() for _ in range(d)] for _ in range(N)]
K = [[random.random() for _ in range(d)] for _ in range(N)]
V = [[random.random() for _ in range(d)] for _ in range(N)]
row, trace = flash_attention_row(Q[0], K, V, block_n=3)
print(trace)
print(row)


## Step 3: compare local naive and flash implementations


In [ ]:
flash_path = REPO / "learning" / "kernel-engineering" / "src" / "flash_attention.py"
spec = importlib.util.spec_from_file_location("flash_attention", flash_path)
flash_module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(flash_module)
O_naive = flash_module.attention_naive(Q, K, V)
O_flash = flash_module.attention_flash(Q, K, V, block_n=3)
max_diff = max(abs(O_naive[i][j] - O_flash[i][j]) for i in range(N) for j in range(d))
print("max_diff", max_diff)
assert max_diff < 1e-9


## Step 4: official source smoke check

This checks the official repository layout without compiling CUDA kernels.


In [ ]:
official_root = REPO / "learning" / "kernel-engineering" / "official" / "repos" / "flash-attention"
files = [
    official_root / "flash_attn" / "flash_attn_interface.py",
    official_root / "csrc" / "flash_attn" / "flash_api.cpp",
    official_root / "hopper" / "softmax.h",
]
for file in files:
    print(file.relative_to(REPO), "exists=", file.exists())
assert all(file.exists() for file in files)


## Closed-book checks

1. Explain why FlashAttention is exact even though it processes K/V in blocks.
2. Explain the purpose of running max `m` and normalizer `l`.
3. Explain why reducing HBM reads/writes matters more than just reducing arithmetic count for this algorithm.
